# 01_01_Data Collection Infrabel

In [1]:
from pathlib import Path
    
import pandas as pd

import infrabel_punctuality as ip

DATA_DIR = Path("../data/raw/")

DATA_INT_DIR = Path("../data/intermediate/")

# Table of Contents  

- [1. Dictionaries of Datasets to Ingest from Infrabel Open Data](#1-dictionaries-of-datasets-to-ingest-from-infrabel-open-data)
    - [1.1. First Dictionary of Datasets](#11-first-dictionary-of-datasets)
    - [1.2. Second Dictionary of Datasets - Punctuality Raw Data](#12-second-dictionary-of-datasets---punctuality-raw-data)  
    
- [2. Infrabel Datasets Ingestion](#2-infrabel-datasets-ingestion)
    - [2.1. Download Configuration and Execution](#21-download-configuration-and-execution)
    - [2.2. Download Verification - Spot Check on Sample Files](#22-download-verification---spot-check-on-sample-files)  

- [3. Concatenation of 24 Monthly Datasets into a Single Raw Punctuality Dataset](#3-concatenation-of-24-monthly-datasets-into-a-single-raw-punctuality-dataset)
    - [3.1. Concatenation into Two Yearly Datasets](#31-concatenation-into-two-yearly-datasets)
    - [3.2. Verification of the Export of the Two Raw Punctuality Files](#32-verification-of-the-export-of-the-two-raw-punctuality-files)
    - [3.3. Concatenation of `punctuality_raw`](#33-concatenation-of-punctuality_raw)
    
- [4. Export to Silver layer](#4-export-to-silver-layer)
    - [4.1. Export Verification](#41-export-verification)


## 1. Dictionaries of Datasets to Ingest from Infrabel Open Data

### 1.1. First Dictionary of Datasets

**First dictionary of datasets** `datasets1`:  

* `operational_pts_railway_raw`: future `dim_station`.  

* `monthly_punctuality_agg`: monthly aggregated Infrabel punctuality measures used as a future **benchmark**. This benchmark will not be loaded into the data warehouse but will be used directly in Power BI for a **final comparaison with the recalculated punctuality measures**.  

* `most_important_incidents_punctuality_sum`: future reference table on punctuality incidents. The Infrabel data in this table is aggregated by day an by incident, then filtered on the rule `SUM(Delay_min) >= 1000` to only keep the most important incidents.

In [2]:

datasets_1 = {
    "operational_pts_railway_raw" : "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/operationele-punten-van-het-netwerk/exports/parquet?lang=en&timezone=Europe%2FBerlin",
    "monthly_punctuality_agg" : "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/belangrijkste-incidenten/exports/parquet?lang=fr&timezone=Europe%2FBerlin",
    "most_important_incidents_punctuality_sum" : "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/belangrijkste-incidenten/exports/parquet?lang=fr&timezone=Europe%2FBerlin"
}

### 1.2. Second Dictionary of Datasets - Punctuality Raw Data

**Second dictionary of datasets** `datasets2`: 

* A dictionary of **24 monthly datasets of raw punctuality data**. They will be ingested and concatenated into the future `fact_punctuality`.  

* The monthly raw punctuality datasets are available on the Infrabel Open Data portal as `.csv` files.  

* In the cell below, the dictionary keys and URLs are generated automatically from a year-month loop, **avoiding hardcoding 24 key-value pairs manually**.

In [3]:
years = [2024, 2025]
months = [f"{m:02d}" for m in range(1, 13)]

datasets_2 = {}

for year in years:
    for month in months:
        key = f"punctuality_raw_{month}{year}"

        url = f"https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_{year}{month}.csv"

        datasets_2[key] = url

for key in datasets_2:
    print(f"{key} : {url} \n")

punctuality_raw_012024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_022024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_032024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_042024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_052024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_062024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_072024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_082024 : https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/Data_raw_punctuality_202512.csv 

punctuality_raw_092024 : https://fr.ftp.opendata

## 2. Infrabel Datasets Ingestion

### 2.1. Download Configuration and Execution

The cell below calls the `run_download()` function from the `infrabel_punctuality` module, passing `datasets1` and the target directory as parameters.

In [4]:
ip.run_download(datasets_1, output_dir=DATA_DIR)
   

Début du téléchargement : operational_pts_railway_raw



operational_pts_railway_raw téléchargé dans ..\data\raw\operational_pts_railway_raw.parquet
Début du téléchargement : monthly_punctuality_agg



monthly_punctuality_agg téléchargé dans ..\data\raw\monthly_punctuality_agg.parquet
Début du téléchargement : most_important_incidents_punctuality_sum



most_important_incidents_punctuality_sum téléchargé dans ..\data\raw\most_important_incidents_punctuality_sum.parquet


**⚠️ Warning**: 
* The cell below downloads 24 raw data files from the Infrabel Open Data portal. Depending on your internet connection, this may take approximately **30 minutes**. 

* The line of code that triggers the download is **commented out for safety**. Uncomment line 3 if you want to start the download.

The `run_download()` function is called, passing `datasets2`, the file type `csv`, the target directory, and the manifest name `manifest_punctuality_raw.txt` as parameters.

In [5]:
ip.run_download(datasets_2, output_dir=DATA_DIR, file_type="csv", registry_name="manifest_punctuality_raw.txt")

Début du téléchargement : punctuality_raw_012024



punctuality_raw_012024 téléchargé dans ..\data\raw\punctuality_raw_012024.csv
Début du téléchargement : punctuality_raw_022024



punctuality_raw_022024 téléchargé dans ..\data\raw\punctuality_raw_022024.csv
Début du téléchargement : punctuality_raw_032024



punctuality_raw_032024 téléchargé dans ..\data\raw\punctuality_raw_032024.csv
Début du téléchargement : punctuality_raw_042024



punctuality_raw_042024 téléchargé dans ..\data\raw\punctuality_raw_042024.csv
Début du téléchargement : punctuality_raw_052024



...

punctuality_raw_082025 téléchargé dans ..\data\raw\punctuality_raw_082025.csv
Début du téléchargement : punctuality_raw_092025



punctuality_raw_092025 téléchargé dans ..\data\raw\punctuality_raw_092025.csv
Début du téléchargement : punctuality_raw_102025



punctuality_raw_102025 téléchargé dans ..\data\raw\punctuality_raw_102025.csv
Début du téléchargement : punctuality_raw_112025



punctuality_raw_112025 téléchargé dans ..\data\raw\punctuality_raw_112025.csv
Début du téléchargement : punctuality_raw_122025



punctuality_raw_122025 téléchargé dans ..\data\raw\punctuality_raw_122025.csv


### 2.2. Download Verification - Spot Check on Sample Files

In [6]:
ope_pts = pd.read_parquet(DATA_DIR / "operational_pts_railway_raw.parquet")
ope_pts.head()

,geo_point_2d,geo_shape,ptcarid,taftapcode,symbolicname,shortnamefrench,shortnamedutch,longnamefrench,longnamedutch,commercialshortnamefrench,commercialshortnamedutch,commercialmiddlenamefrench,commercialmiddlenamedutch,commerciallongnamefrench,commerciallongnamedutch,class_en
0,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,443,BE00443,FKGLF,GENK-ZD-FORD,GENK-ZD-FORD,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Genk-Zd-Ford,Genk-Zd-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Station
1,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,275,BE00275,GCRA,CHARLEROI-AT,CHARLEROI-AT,CHARLEROI-A.T.,CHARLEROI-A.T.,Charleroi-AT,Charleroi-AT,Charleroi-A.T.,Charleroi-A.T.,Charleroi-A.T.,Charleroi-A.T.,Service installation
2,"b'\x01\x01\x00\x00\x00>\x84\x92P\xf8,\x11@\x15...","b'\x01\x01\x00\x00\x00>\x84\x92P\xf8,\x11@\x15...",1748,BE01748,VOIL1,V.OILTANK1,V.OILTANK1,VERB.OILTANKING 1,VERB.OILTANKING 1,V.Oiltank1,V.Oiltank1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verb.Oiltanking 1,Connection
3,b'\x01\x01\x00\x00\x00/J\x7f\xef4)\x11@o\xf1\x...,b'\x01\x01\x00\x00\x00/J\x7f\xef4)\x11@o\xf1\x...,1749,BE01749,VOIL2,V.OILTANK2,V.OILTANK2,VERB.OILTANKING 2,VERB.OILTANKING 2,V.Oiltank2,V.Oiltank2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verb.Oiltanking 2,Connection
4,"b'\x01\x01\x00\x00\x00\x93\xbc\xb4\xbeF""\x11@7...","b'\x01\x01\x00\x00\x00\x93\xbc\xb4\xbeF""\x11@7...",1751,BE01751,VIBR,V.IBR,V.IBR,VERB.IBR,VERB.IBR,V.IBR,V.IBR,Verb.IBR,Verb.IBR,Verb.IBR,Verb.IBR,Connection


In [7]:
punctuality_monthly_agg = pd.read_parquet(DATA_DIR / "monthly_punctuality_agg.parquet")
punctuality_monthly_agg.head()

,month,date,lijn,lieu,description_de_l_incident,min_delay,sup
0,2026-04-01,2026-04-29,124,LINKEBEEK,Heurt d'une personne,1653,83
1,2026-04-01,2026-04-28,25,ANTWERPEN-CENTRAAL,Intrusion dans les voies,1103,11
2,2026-04-01,2026-04-18,96,HALLE,Dérangement à la signalisation,1366,34
3,2026-04-01,2026-04-01,19,MOL,Dérangement à l'infrastructure,1172,15
4,2026-03-01,2026-03-27,161,CHASTRE,Heurt d'une personne,1501,64


In [8]:
punctuality_monthly_raw = pd.read_csv(DATA_DIR / "punctuality_raw_012024.csv")
punctuality_monthly_raw.head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,THOP1_COD,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,...,DELAY_ARR,DELAY_DEP,CIRC_TYP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP
0,01JAN2024,1813,IC 02,SNCB/NMBS,929,NaN,50A,NaN,13:09:42,NaN,...,NaN,42.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,NaN,NaN,01JAN2024,NaN,01JAN2024
1,01JAN2024,1813,IC 02,SNCB/NMBS,210,=,50A,13:22:17,13:25:20,13:22:00,...,17.0,20.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024
2,01JAN2024,1813,IC 02,SNCB/NMBS,931,D,50A,13:29:18,13:29:18,13:28:00,...,78.0,78.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024
3,01JAN2024,1813,IC 02,SNCB/NMBS,127,P,50A,13:31:59,13:31:59,13:31:00,...,59.0,59.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024
4,01JAN2024,1813,IC 02,SNCB/NMBS,797,D,50A,13:33:57,13:33:57,13:33:00,...,57.0,57.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024


## 3. Concatenation of 24 Monthly Datasets into a Single Raw Punctuality Dataset

* The 24 monthly raw datasets are concatenated first into **two yearly datasets**, then into **a single dataset**.  

* This dataset will serve as the source for the data warehouse's `fact_punctuality`.

### 3.1. Concatenation into Two Yearly Datasets

The monthly punctuality datasets paths are collected into two yearly lists - `files_2024` and `files_2025` - then sorted.

In [9]:
files_2024 = list(DATA_DIR.glob("punctuality_raw_*2024.csv"))
files_2024.sort()

files_2025 = list(DATA_DIR.glob("punctuality_raw_*2025.csv"))
files_2025.sort()

* The twelve 2024 monthly datasets are concatenated into a single DataFrame - `punctuality_raw_2024`.  

* `punctuality_raw_2024` is exported as a `.parquet` file.

* `punctuality_raw_2024` is deleted to free RAM.

* Then, the same operations are executed for the twelve 2025 monthly datasets and a `punctuality_raw_2025` file is exported.

**⚠️ Warning**: The cell below may take **3 to 4 minutes** to execute.

In [10]:
punctuality_raw_2024 = pd.concat(
    (pd.read_csv(f) for f in files_2024),
    ignore_index=True
)
punctuality_raw_2024.head()

punctuality_raw_2024.to_parquet(DATA_INT_DIR / "punctuality_raw_2024.parquet")

del punctuality_raw_2024

df_punctuality_raw_2025 = pd.concat(
    (pd.read_csv(f) for f in files_2025),
    ignore_index=True
)
df_punctuality_raw_2025.head()

df_punctuality_raw_2025.to_parquet(DATA_INT_DIR / "punctuality_raw_2025.parquet")

del df_punctuality_raw_2025


### 3.2. Verification of the Export of the Two Raw Punctuality Files

In [11]:
punct_2024 = pd.read_parquet(DATA_INT_DIR / "punctuality_raw_2024.parquet")
punct_2024.head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,THOP1_COD,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,...,DELAY_ARR,DELAY_DEP,CIRC_TYP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP
0,01JAN2024,1813,IC 02,SNCB/NMBS,929,NaN,50A,NaN,13:09:42,NaN,...,NaN,42.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,NaN,NaN,01JAN2024,NaN,01JAN2024
1,01JAN2024,1813,IC 02,SNCB/NMBS,210,=,50A,13:22:17,13:25:20,13:22:00,...,17.0,20.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024
2,01JAN2024,1813,IC 02,SNCB/NMBS,931,D,50A,13:29:18,13:29:18,13:28:00,...,78.0,78.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024
3,01JAN2024,1813,IC 02,SNCB/NMBS,127,P,50A,13:31:59,13:31:59,13:31:00,...,59.0,59.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024
4,01JAN2024,1813,IC 02,SNCB/NMBS,797,D,50A,13:33:57,13:33:57,13:33:00,...,57.0,57.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024


In [12]:
punct_2025 = pd.read_parquet(DATA_INT_DIR / "punctuality_raw_2025.parquet")
punct_2025.head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,THOP1_COD,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,...,DELAY_DEP,CIRC_TYP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,OP1_COD
0,01JAN2025,3117,IC 31,SNCB/NMBS,259,NaN,124,NaN,17:05:40,NaN,...,40.0,1,IC 31: CHARLEROI-CENTRAL -> ANTWERPEN-CENTRAAL,CHARLEROI-CENTRAL,NaN,NaN,01JAN2025,NaN,01JAN2025,NaN
1,01JAN2025,3117,IC 31,SNCB/NMBS,791,=,124,17:09:50,17:11:27,17:10:00,...,27.0,1,IC 31: CHARLEROI-CENTRAL -> ANTWERPEN-CENTRAAL,MARCHIENNE-AU-PONT,124,01JAN2025,01JAN2025,01JAN2025,01JAN2025,NaN
2,01JAN2025,3117,IC 31,SNCB/NMBS,1018,P,124,17:14:29,17:14:29,17:14:00,...,29.0,1,IC 31: CHARLEROI-CENTRAL -> ANTWERPEN-CENTRAAL,ROUX,124,01JAN2025,01JAN2025,01JAN2025,01JAN2025,NaN
3,01JAN2025,3117,IC 31,SNCB/NMBS,286,D,124,17:15:39,17:15:39,17:15:00,...,39.0,1,IC 31: CHARLEROI-CENTRAL -> ANTWERPEN-CENTRAAL,COURCELLES-MOTTE,124,01JAN2025,01JAN2025,01JAN2025,01JAN2025,NaN
4,01JAN2025,3117,IC 31,SNCB/NMBS,768,=,124,17:19:02,17:20:18,17:19:00,...,18.0,1,IC 31: CHARLEROI-CENTRAL -> ANTWERPEN-CENTRAAL,LUTTRE,124,01JAN2025,01JAN2025,01JAN2025,01JAN2025,NaN


### 3.3. Concatenation of `punctuality_raw`

The two yearly raw punctuality datasets are concatenated into a single `punctuality_raw` DataFrame, then exported as a `.parquet` file in the next section.

In [13]:
punctuality_raw = pd.concat([punct_2024, punct_2025], ignore_index=True)

## 4. Export to Silver Layer

**⚠️ Warning**: The cell below may take **more than a minute** to execute.

In [14]:
punctuality_raw.to_parquet(DATA_INT_DIR / "punctuality_raw.parquet")

del punctuality_raw

### 4.1. Export Verification

In [15]:
df_to_check = pd.read_parquet(DATA_INT_DIR / "punctuality_raw.parquet")
df_to_check.head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,THOP1_COD,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,...,DELAY_DEP,CIRC_TYP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,OP1_COD
0,01JAN2024,1813,IC 02,SNCB/NMBS,929,NaN,50A,NaN,13:09:42,NaN,...,42.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,NaN,NaN,01JAN2024,NaN,01JAN2024,NaN
1,01JAN2024,1813,IC 02,SNCB/NMBS,210,=,50A,13:22:17,13:25:20,13:22:00,...,20.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
2,01JAN2024,1813,IC 02,SNCB/NMBS,931,D,50A,13:29:18,13:29:18,13:28:00,...,78.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
3,01JAN2024,1813,IC 02,SNCB/NMBS,127,P,50A,13:31:59,13:31:59,13:31:00,...,59.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
4,01JAN2024,1813,IC 02,SNCB/NMBS,797,D,50A,13:33:57,13:33:57,13:33:00,...,57.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
